# EDA — Jugadores, 5 grandes ligas (Understat)

Datos ofensivos de jugadores (goles, asistencias, xG, xA y sus tasas por 90')
de las 5 grandes ligas europeas, temporada 2025-26, vía Understat. A
diferencia de `eda_teams.ipynb` (fbref, nivel equipo), esta base es a nivel
**jugador** — menos columnas para explotar, pero permite ver quién sobre o
bajo-rendimió su expected goals/assists, y comparar perfiles ofensivos
individuales entre ligas.

In [1]:
import html
import sys

import pandas as pd

sys.path.insert(0, ".")
from viz_theme import (
    LEAGUE_ORDER, apply_theme, apply_plotly_theme, league_color,
    render_with_sidebar, render_plot, league_box_figure,
)

apply_theme()
apply_plotly_theme()

## Carga y limpieza

Un CSV por liga (`;` como separador), sin columna de liga explícita — se
asigna por archivo. Dos limpiezas puntuales sobre `player`/`team`:

- Algunos nombres traen entidades HTML sin decodificar (ej. `M&#039;Bala
  Nzola`) — se decodifican con `html.unescape`.
- Jugadores transferidos a mitad de temporada traen los dos equipos
  concatenados en `team` (ej. `"Angers,Rennes"`) — nos quedamos con el
  **último** equipo.

In [2]:
SEASON = "2025-26"

FILES = {
    "bundes-players.csv": "Bundesliga",
    "seriea-players.csv": "Serie A",
    "ligue1-players.csv": "Ligue 1",
    "laliga-players.csv": "La Liga",
    "premier-players.csv": "Premier League",
}

dfs = []
for fname, liga in FILES.items():
    d = pd.read_csv(f"../data/{SEASON}/{fname}", sep=";", encoding="utf-8")
    d["liga"] = liga
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)

df["player"] = df["player"].apply(html.unescape)
df["team"] = df["team"].str.split(",").str[-1].str.strip()

print(f"{len(df)} jugadores en las 5 ligas")
df.head()

2775 jugadores en las 5 ligas


,number,player,team,apps,min,goals,a,xG,xA,xG90,xA90,liga
0,1,Harry Kane,Bayern Munich,31,2385,36,5,29.58,7.40,1.12,0.28,Bundesliga
1,2,Deniz Undav,VfB Stuttgart,29,2261,19,6,19.06,5.58,0.76,0.22,Bundesliga
2,3,Serhou Guirassy,Borussia Dortmund,33,2366,17,1,19.91,1.93,0.76,0.07,Bundesliga
3,4,Patrik Schick,Bayer Leverkusen,28,2005,16,3,19.42,3.19,0.87,0.14,Bundesliga
4,5,Michael Olise,Bayern Munich,32,2308,15,19,11.55,17.04,0.45,0.66,Bundesliga


### Filtro de minutos mínimos

`xG90`/`xA90` son tasas: con pocos minutos jugados son ruido puro (un
suplente con 200 minutos y una racha de suerte puede tener una tasa
disparatada). Nos quedamos con jugadores con **≥900 minutos** (~10 partidos
completos) para todo lo que sigue — scatters y boxplots.

In [3]:
MIN_MINUTES = 900

before = len(df)
df = df[df["min"] >= MIN_MINUTES].reset_index(drop=True)
print(f"filtro >= {MIN_MINUTES} min: {before} -> {len(df)} jugadores")
df.groupby("liga").size().reindex(LEAGUE_ORDER)

filtro >= 900 min: 2775 -> 1583 jugadores


liga
Bundesliga        284
Serie A           338
Ligue 1           276
La Liga           345
Premier League    340
dtype: int64

## 1. Producción vs. expected

Tres scatters interactivos, un punto por jugador — pasa el mouse para ver
el detalle, busca un jugador o filtra por liga en la barra lateral.

### Goles vs. xG

Quién sobre o bajo-rendimió su expected goals. La línea punteada es
`goles = xG`: por encima, el jugador metió más de lo que sus ocasiones
merecían (mejor definición o algo de suerte); por debajo, le faltó pegada
o tuvo mala fortuna.

In [4]:
import plotly.graph_objects as go

mx = max(df["goals"].max(), df["xG"].max()) * 1.05

# Nota: la(s) traza(s) "extra" (ej. la línea de referencia) deben agregarse
# DESPUÉS de las trazas de liga — render_with_sidebar asume ese orden para
# no desalinear los índices de traza al filtrar/buscar.
fig = go.Figure()

scatter_data = []
for liga in LEAGUE_ORDER:
    sub = df[df["liga"] == liga]
    scatter_data.append((liga, sub))
    fig.add_trace(go.Scatter(
        x=sub["xG"], y=sub["goals"], mode="markers", name=liga,
        marker=dict(color=league_color(liga), size=8, opacity=0.75,
                    line=dict(width=0.5, color="white")),
        customdata=sub[["player", "team"]],
        hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                       "xG: %{x:.1f}<br>Goles: %{y:.0f}<extra></extra>",
    ))

fig.add_trace(go.Scatter(
    x=[0, mx], y=[0, mx], mode="lines",
    line=dict(color="#c3c2b7", dash="dash", width=1.4),
    hoverinfo="skip", showlegend=False,
))

fig.update_layout(
    title=dict(text="Goles vs. xG",
               subtitle=dict(text="¿Quién sobre/bajo-rindió su expected goals? "
                                   f"Jugadores con ≥{MIN_MINUTES} min, 2025-26")),
    xaxis_title="xG", yaxis_title="Goles",
)

render_with_sidebar(fig, scatter_data, "xG", "goals", extra_traces=1,
                     name_col="player", search_label="jugador",
                     width=760, height=580)

### Asistencias vs. xA

Mismo concepto para creación de juego: quién generó más o menos asistencias
de las que sus pases de gol merecían.

In [5]:
mx2 = max(df["a"].max(), df["xA"].max()) * 1.05

fig2 = go.Figure()

scatter_data2 = []
for liga in LEAGUE_ORDER:
    sub = df[df["liga"] == liga]
    scatter_data2.append((liga, sub))
    fig2.add_trace(go.Scatter(
        x=sub["xA"], y=sub["a"], mode="markers", name=liga,
        marker=dict(color=league_color(liga), size=8, opacity=0.75,
                    line=dict(width=0.5, color="white")),
        customdata=sub[["player", "team"]],
        hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                       "xA: %{x:.1f}<br>Asistencias: %{y:.0f}<extra></extra>",
    ))

fig2.add_trace(go.Scatter(
    x=[0, mx2], y=[0, mx2], mode="lines",
    line=dict(color="#c3c2b7", dash="dash", width=1.4),
    hoverinfo="skip", showlegend=False,
))

fig2.update_layout(
    title=dict(text="Asistencias vs. xA",
               subtitle=dict(text="¿Quién sobre/bajo-rendimió su expected assists? "
                                   f"Jugadores con ≥{MIN_MINUTES} min, 2025-26")),
    xaxis_title="xA", yaxis_title="Asistencias",
)

render_with_sidebar(fig2, scatter_data2, "xA", "a", extra_traces=1,
                     name_col="player", search_label="jugador",
                     width=760, height=580)

### Perfil ofensivo: xG90 vs. xA90

Tasas por 90 minutos en vez de totales, para no mezclar producción con
minutos jugados. Separa perfiles: killers puros (abajo a la derecha),
creadores puros (arriba a la izquierda) y todocampo (arriba a la
derecha) — las líneas punteadas marcan el promedio de cada eje.

In [6]:
fig3 = go.Figure()

scatter_data3 = []
for liga in LEAGUE_ORDER:
    sub = df[df["liga"] == liga]
    scatter_data3.append((liga, sub))
    fig3.add_trace(go.Scatter(
        x=sub["xG90"], y=sub["xA90"], mode="markers", name=liga,
        marker=dict(color=league_color(liga), size=8, opacity=0.75,
                    line=dict(width=0.5, color="white")),
        customdata=sub[["player", "team"]],
        hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                       "xG90: %{x:.2f}<br>xA90: %{y:.2f}<extra></extra>",
    ))

xm, ym = df["xG90"].mean(), df["xA90"].mean()
fig3.add_hline(y=ym, line=dict(color="#c3c2b7", width=1, dash="dot"))
fig3.add_vline(x=xm, line=dict(color="#c3c2b7", width=1, dash="dot"))

fig3.update_layout(
    title=dict(text="Perfil ofensivo: xG90 vs. xA90",
               subtitle=dict(text="Killer puro, creador puro o todocampo — "
                                   f"jugadores con ≥{MIN_MINUTES} min, 2025-26")),
    xaxis_title="xG por 90'", yaxis_title="xA por 90'",
)

render_with_sidebar(fig3, scatter_data3, "xG90", "xA90",
                     name_col="player", search_label="jugador",
                     width=760, height=580)

## 2. Nivel ofensivo por liga

Las tasas por 90' de todos los jugadores con minutos suficientes, liga por
liga — para comparar el nivel goleador y creativo entre las 5 ligas, no
solo entre jugadores individuales.

In [7]:
figb1 = league_box_figure(df, "xG90", "xG por 90'", hover_fmt=".2f", name_col="player")
figb1.update_layout(
    title=dict(text="Nivel goleador esperado por liga",
               subtitle=dict(text=f"xG por 90' — jugadores con ≥{MIN_MINUTES} min, 2025-26")),
    yaxis_title="xG90",
)
render_plot(figb1, width=800, height=520)

In [8]:
figb2 = league_box_figure(df, "xA90", "xA por 90'", hover_fmt=".2f", name_col="player")
figb2.update_layout(
    title=dict(text="Nivel de creación esperado por liga",
               subtitle=dict(text=f"xA por 90' — jugadores con ≥{MIN_MINUTES} min, 2025-26")),
    yaxis_title="xA90",
)
render_plot(figb2, width=800, height=520)